# mimic-iv: OpenICU → YAIB wide

This is the complete dataset workflow. It writes the all-concept OpenICU wide table below the OpenICU workspace `yaib/mimic-iv/` directory. For RICU validation it additionally creates OpenICU and RICU 7-day wide tables and comparison reports.


In [1]:
from pathlib import Path
import os
import subprocess

from openicu_yaib import (
    build_and_write_yaib_wide_for_dataset,
    compare_openicu_wide_to_ricu_for_dataset,
    concept_root_from_output,
    resolve_openicu_workspace,
    write_all_concepts_wide,
)

DATASET = "mimic-iv"
MAX_HOURS = 7 * 24

# Point this at either the OpenICU project root, its workspace directory,
# the concept directory, or a separate output directory with concept/ below it.
OPENICU_OUTPUT = Path.home() / "output" / "OpenICU.example" / "output" / "project" / "workspace"
WORKSPACE = resolve_openicu_workspace(OPENICU_OUTPUT)
CONCEPT_ROOT = concept_root_from_output(OPENICU_OUTPUT)
DATASET_OUTPUT = WORKSPACE / "yaib" / DATASET
DATASET_OUTPUT.mkdir(parents=True, exist_ok=True)

# Optional override if automatic raw stay-table discovery is not sufficient:
STAYS_PATH = None
RICU_CONCEPT_DICT = Path.home() / "workspace" / "ricu" / "inst" / "extdata" / "config" / "concept-dict.json"


In [2]:
# 1) Full OpenICU YAIB-wide table across every available concept.
all_result = write_all_concepts_wide(
    dataset=DATASET,
    openicu_output=OPENICU_OUTPUT,
    concept_root=CONCEPT_ROOT,
    stays_path=STAYS_PATH,
    max_hours=None,
)
all_result


AllConceptsExportResult(output_path=PosixPath('/home/q039tl/output/OpenICU.example/output/project/workspace/yaib/mimic-iv/openicu_all_concepts_wide.parquet'), manifest_path=PosixPath('/home/q039tl/output/OpenICU.example/output/project/workspace/yaib/mimic-iv/openicu_all_concepts_wide_concepts.csv'), concepts=('CO2_partial_pressure', 'C_reactive_protein', 'GCS_eye', 'GCS_motor', 'GCS_verbal', 'Hemoglobin_A1C', 'O2_partial_pressure', 'Richmond_agitation_sedation_scale', 'alanine_aminotransferase', 'albumin', 'alkaline_phosphatase', 'antibiotics', 'arterial_oxygen_saturation', 'aspartate_aminotransferase', 'band_form_neutrophils', 'base_excess', 'basophils', 'bicarbonate', 'bilirubin_direct', 'blood_urea_nitrogen', 'body_fluid_sampling', 'calcium', 'calcium_ionized', 'chloride', 'creatine_kinase', 'creatine_kinase_MB', 'creatinine', 'dextrose_as_D10', 'diastolic_blood_pressure', 'dobutamine_duration', 'dobutamine_rate', 'dopamine_duration', 'dopamine_rate', 'endtidal_CO2', 'eosinophils', 

In [3]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path("/home/q039tl/workspace/openicu-yaib")

env = os.environ.copy()
env["RICU_OUT_DIR"] = str(DATASET_OUTPUT)
env["RICU_DATA_PATH"] = "/home/q039tl/ricu_data"
env["RICU_SRC_LOAD"] = (
    "mimic,mimic_demo,eicu,eicu_demo,hirid,aumc,miiv,sic"
)
env.pop("RICU_CONFIG_PATH", None)
env["R_ENVIRON_USER"] = "/dev/null"

subprocess.run(
    [
        "Rscript",
        "--no-environ",
        "scripts/datasets/export_ricu_mimic-iv.R",
    ],
    cwd=REPO_ROOT,
    check=True,
    env=env,
)

── Loading 48 concepts ─────────────────────────────────────────────────────────
• alb
  ◯ removed 69 (0.04%) of rows due to `NA` values
  ◯ removed 7 (0%) of rows due to out of range entries
• alp
  ◯ removed 59 (0.02%) of rows due to `NA` values
• alt
  ◯ removed 4733 (1.48%) of rows due to `NA` values
• ast
  ◯ removed 288 (0.09%) of rows due to `NA` values
• be
  ◯ removed 617 (0.11%) of rows due to `NA` values
  ◯ removed 588 (0.1%) of rows due to out of range entries
• bicar
  ◯ removed 406 (0.04%) of rows due to `NA` values
  ◯ removed 118 (0.01%) of rows due to out of range entries
• bili
  ◯ removed 4402 (1.38%) of rows due to `NA` values
• bili_dir
  ◯ removed 1072 (5.37%) of rows due to `NA` values
  ◯ removed 3 (0.02%) of rows due to out of range entries
• bnd
  ◯ removed 10 (0.02%) of rows due to `NA` values
• bun
  ◯ removed 777 (0.07%) of rows due to `NA` values
  ◯ removed 322 (0.03%) of rows due to out of range entries
• ca
  ◯ removed 162 (0.02%) of rows due to `NA` v

 [1] "stay_id"   "charttime" "alb"       "alp"       "alt"       "ast"      
 [7] "be"        "bicar"     "bili"      "bili_dir"  "bnd"       "bun"      
[13] "ca"        "cai"       "ck"        "ckmb"      "cl"        "crea"     
[19] "crp"       "dbp"       "fgn"       "fio2"      "glu"       "hgb"      
[25] "hr"        "inr_pt"    "k"         "lact"      "lymph"     "map"      
[31] "mch"       "mchc"      "mcv"       "methb"     "mg"        "na"       
[37] "neut"      "o2sat"     "pco2"      "ph"        "phos"      "plt"      
[43] "po2"       "ptt"       "resp"      "sbp"       "temp"      "tnt"      
[49] "urine"     "wbc"      
Key: <stay_id, charttime>
    stay_id  charttime   alb   alp   alt   ast    be bicar  bili bili_dir   bnd
      <int> <difftime> <num> <num> <num> <num> <num> <num> <num>    <num> <num>
1: 30000153   -1 hours    NA    NA    NA    NA    NA    NA    NA       NA    NA
2: 30000153    0 hours    NA    NA    NA    NA    NA    NA    NA       NA    NA
3: 300001

Wrote parquet: /home/q039tl/output/OpenICU.example/output/project/workspace/yaib/mimic-iv/ricu_dynamic_vars_miiv.parquet
Wrote parquet: /home/q039tl/output/OpenICU.example/output/project/workspace/yaib/mimic-iv/ricu_stay_windows_miiv.parquet


[1] "stay_id" "start"   "end"    
Key: <stay_id>
    stay_id      start        end
      <int> <difftime> <difftime>
1: 30000153    0 hours   39 hours
2: 30000213    0 hours   39 hours
3: 30000484    0 hours   59 hours
4: 30000646    0 hours  112 hours
5: 30000831    0 hours   64 hours
6: 30001148    0 hours   27 hours
[1] "stay_id"


CompletedProcess(args=['Rscript', '--no-environ', 'scripts/datasets/export_ricu_mimic-iv.R'], returncode=0)

In [4]:
# 3) OpenICU YAIB-wide table for exactly the first 7 days.
openicu_7d = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    version="1.0.0",
)
openicu_7d


WideExportResult(output_path=PosixPath('/home/q039tl/output/OpenICU.example/output/project/workspace/yaib/mimic-iv/openicu_dyn_168h.parquet'), summary=shape: (1, 4)
┌─────────┬─────────┬──────────┬──────────┐
│ n_rows  ┆ n_stays ┆ min_time ┆ max_time │
│ ---     ┆ ---     ┆ ---      ┆ ---      │
│ u32     ┆ u32     ┆ i64      ┆ i64      │
╞═════════╪═════════╪══════════╪══════════╡
│ 6269536 ┆ 94458   ┆ 0        ┆ 168      │
└─────────┴─────────┴──────────┴──────────┘)

In [5]:
# 4) Compare only the 7-day OpenICU and RICU wide representations.
comparison = compare_openicu_wide_to_ricu_for_dataset(
    dataset=DATASET,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
)
comparison.as_dict()


{'table_summary': shape: (2, 5)
 ┌────────────────┬─────────┬─────────┬──────────┬──────────┐
 │ table          ┆ n_rows  ┆ n_stays ┆ min_time ┆ max_time │
 │ ---            ┆ ---     ┆ ---     ┆ ---      ┆ ---      │
 │ str            ┆ u32     ┆ u32     ┆ i64      ┆ i64      │
 ╞════════════════╪═════════╪═════════╪══════════╪══════════╡
 │ openicu        ┆ 6267170 ┆ 94444   ┆ 0        ┆ 168      │
 │ ricu_reference ┆ 5995769 ┆ 94434   ┆ 0        ┆ 168      │
 └────────────────┴─────────┴─────────┴──────────┴──────────┘,
 'stay_overlap': shape: (1, 5)
 ┌─────────────────┬───────────────────┬────────────────┬─────────────────────┬─────────────────────┐
 │ n_openicu_stays ┆ n_reference_stays ┆ n_common_stays ┆ n_only_openicu_stay ┆ n_only_reference_st │
 │ ---             ┆ ---               ┆ ---            ┆ s                   ┆ ays                 │
 │ i64             ┆ i64               ┆ i64            ┆ ---                 ┆ ---                 │
 │                 ┆            